In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
    
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
def create_features(X):
        X = X.copy()

        X['PregnancyGroup'] = pd.cut(
        X['Pregnancies'],
        bins=[-1,0,3,6,20],
       labels=['0','1-3','4-6','7+'])
    
        X['AgeGroup'] = pd.cut(
            X['Age'],
            bins=[20,30,40,50,60,80],
            labels=['20-30','31-40','41-50','51-60','61-80'],
        )
        
        X['Glucose_BMI'] = X['Glucose']*X['BMI']
    
        return X
    
feature_creator = FunctionTransformer(create_features)

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [4]:
df = pd.read_csv("cleaned_data_after_eda.csv")



print(df.info())
X = df.drop("Outcome", axis=1)
y = df["Outcome"]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 766 entries, 0 to 765
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               766 non-null    int64  
 1   Glucose                   766 non-null    float64
 2   BloodPressure             766 non-null    float64
 3   SkinThickness             766 non-null    float64
 4   Insulin                   766 non-null    float64
 5   BMI                       766 non-null    float64
 6   DiabetesPedigreeFunction  766 non-null    float64
 7   Age                       766 non-null    int64  
 8   Outcome                   766 non-null    int64  
 9   BMI_Category              766 non-null    object 
 10  AgeGroup                  765 non-null    object 
 11  PregnancyGroup            766 non-null    object 
 12  Cluster                   766 non-null    int64  
dtypes: float64(6), int64(4), object(3)
memory usage: 77.9+ KB
None


In [5]:
X_train, X_test, y_train, y_test = train_test_split(    X,y,
  test_size=0.2,random_state=42,    stratify=y)

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import FunctionTransformer

num_features = [
    'Pregnancies','Glucose','BloodPressure',
    'SkinThickness','Insulin','BMI',
    'DiabetesPedigreeFunction','Age','Glucose_BMI'
    ]
    
cat_features = ['PregnancyGroup','AgeGroup']
    
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), num_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
        ])

In [7]:

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
    
scoring='recall_macro'

pipeline = Pipeline([
        ('features', FunctionTransformer(create_features)),
        ('preprocessing', preprocessor),('smote',SMOTE()),
        ('model', LogisticRegression())  # placeholder model
])

In [8]:
pos_weight = sum(y_train == 0) / sum(y_train == 1)

param_grid = [
    {
        "model": [LogisticRegression(max_iter=1000, random_state=42)],
        "model__C": [0.01, 0.1, 1, 10],
        "model__class_weight": [None, "balanced"]
    },
    {
        "model": [RandomForestClassifier(random_state=42)],
        "model__n_estimators": [50, 100, 200, 500],
        "model__min_samples_split": [5, 50, 100],
        "model__class_weight": [None, "balanced"]
    },
    {
        "model": [SVC(probability=True, random_state=42)],
        "model__C": [0.1, 1, 10, 50],
        "model__class_weight": [None, "balanced"]
    },
    {
        "model": [XGBClassifier(
            eval_metric="logloss",
            random_state=42
        )],
        "model__n_estimators": [10, 50, 100, 500],
        "model__scale_pos_weight": [
            1,
            sum(y_train == 0) / sum(y_train == 1)
        ]
    },
    {
        "model": [DecisionTreeClassifier(random_state=42)],
        "model__max_depth": [3, 10, 50, None],
        "model__class_weight": [None, "balanced"]
    }
]

In [9]:


model = XGBClassifier(
    base_score=None, booster=None, callbacks=None,
    colsample_bylevel=None, colsample_bynode=None,
    colsample_bytree=None, device=None, early_stopping_rounds=None,
    enable_categorical=False, eval_metric='logloss',
    feature_types=None, feature_weights=None, gamma=None,
    grow_policy=None, importance_type=None,
    interaction_constraints=None, learning_rate=None, max_bin=None,
    max_cat_threshold=None, max_cat_to_onehot=None,
    max_delta_step=None, max_depth=None, max_leaves=None,
    min_child_weight=None, missing=np.nan, monotone_constraints=None,
    multi_strategy=None, n_estimators=50, n_jobs=None,
    num_parallel_tree=None, scale_pos_weight=1
)

In [10]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=pipeline,  
    param_grid=param_grid,
    cv=5,
    scoring=scoring,      
    n_jobs=-1
)

grid.fit(X_train,y_train)

print(grid.best_params_)
y_pred = grid.predict(X_test)


{'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...), 'model__n_estimators': 100, 'model__scale_pos_weight': 1}


In [11]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
from sklearn.metrics import accuracy_score, precision_score, recall_score
    
print("Accuracy:" ,accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall:" ,recall_score(y_test,y_pred))

Accuracy: 0.8831168831168831
Precision: 0.8103448275862069
Recall: 0.8703703703703703


In [12]:
from imblearn.pipeline import Pipeline

from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
    
scoring = 'recall' 

param_grid = {
        'model__n_estimators': [200, 300, 500],
       'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.01, 0.05, 0.1],
       'model__subsample': [0.7, 0.8, 1.0],
       'model__colsample_bytree': [0.7, 0.8, 1.0],
       'model__scale_pos_weight': [1.0, 1.5, 1.8689, 2.0]}
    
pipeline = Pipeline([
       ('features', FunctionTransformer(create_features)),
       ('preprocessing', preprocessor), # encoding/scaling if needed\n",
        ('smote', SMOTE(random_state=42)),
       ('model', XGBClassifier(
           base_score=0.5,
           eval_metric='logloss',
           random_state=42,
           use_label_encoder=False
       ))
    ])
   
grid = RandomizedSearchCV(
       pipeline,
       param_distributions=param_grid,
       n_iter=20,  
        scoring='recall',
        n_jobs=-1,
       verbose=2,
        random_state=42)




    



    
grid.fit(X_train, y_train)
    
    
y_pred = grid.predict(X_test)

print("=== Default threshold (0.5) ===")

print("Default threshold:")

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
    
    # ---------------- Threshold tuning ----------------
y_probs = grid.predict_proba(X_test)[:, 1]
    

threshold = 0.45  # slightly lower for better recall (medical use)
y_pred_thresh = (y_probs >= threshold).astype(int)

print("=== After threshold adjustment ===")
print(classification_report(y_test, y_pred_thresh))
print(confusion_matrix(y_test, y_pred_thresh))

threshold = 0.48
y_pred_thresh = (y_probs >= threshold).astype(int)
print("After threshold adjustment:")
print(classification_report(y_test, y_pred_thresh))
print(confusion_matrix(y_test, y_pred_thresh)) 


Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\sibus\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:05:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== Default threshold (0.5) ===
Default threshold:
              precision    recall  f1-score   support

           0       0.94      0.85      0.89       100
           1       0.77      0.91      0.83        54

    accuracy                           0.87       154
   macro avg       0.86      0.88      0.86       154
weighted avg       0.88      0.87      0.87       154

[[85 15]
 [ 5 49]]
=== After threshold adjustment ===
              precision    recall  f1-score   support

           0       0.97      0.85      0.90       100
           1       0.77      0.94      0.85        54

    accuracy                           0.88       154
   macro avg       0.87      0.90      0.88       154
weighted avg       0.90      0.88      0.89       154

[[85 15]
 [ 3 51]]
After threshold adjustment:
              precision    recall  f1-score   support

           0       0.94      0.85      0.89       100
           1       0.77      0.91      0.83        54

    accuracy                  

In [13]:
import joblib
best_model = grid.best_estimator_
joblib.dump(best_model, "diabetes_model_only.pkl")

['diabetes_model_only.pkl']